# Block 2: LSTM for Multi-Step Time Series Forecasting
## Hochfrequenz AI Workshop

**Objective:** Build LSTM models to predict next 24 hours of household electricity consumption

**Key Concepts:** ANNs → RNNs → LSTMs, Sequence modeling, Multi-step forecasting

**Models:** LSTM (single-step) → LSTM (multi-step) → Compare with Block 1 XGBoost

## Theory: From ANNs to LSTMs

### Why Standard Neural Networks Fail at Time Series

**Problem:** Traditional ANNs (Artificial Neural Networks) treat each input independently:
- They don't remember previous inputs
- Can't capture temporal dependencies
- Example: Predicting consumption at 6 PM requires knowing consumption at 5 PM, 4 PM, etc.

### Recurrent Neural Networks (RNNs)

**Solution:** RNNs maintain a "hidden state" that carries information from previous time steps
- Each time step updates the hidden state
- Hidden state = memory of what came before

**Problem with basic RNNs:** Vanishing gradient problem
- Can't remember patterns from distant past (e.g., 24 hours ago)
- Gradients become too small during backpropagation

### LSTM (Long Short-Term Memory)

**Solution:** LSTMs use gates to control what to remember and forget:

1. **Forget Gate:** Decides what to discard from memory
2. **Input Gate:** Decides what new information to store
3. **Output Gate:** Decides what to output from memory
4. **Cell State:** Long-term memory (can carry information across many time steps)

**Why LSTMs Win for Energy Forecasting:**
- Remember daily patterns (consumption 24h ago)
- Capture weekly patterns (weekday vs weekend)
- Learn seasonal trends
- Handle multiple time scales simultaneously

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# TensorFlow/Keras imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print(f"✅ All libraries imported successfully")
print(f"TensorFlow version: {tf.__version__}")

### 1.1 Load Data (Recap from Block 1)

We'll use the same German household consumption data, but prepare it differently for LSTM (sequences instead of feature vectors)

In [ ]:
# Generate realistic synthetic German household consumption data
np.random.seed(42)

# Generate 2 years of hourly data
date_range = pd.date_range(start='2018-05-01', end='2020-12-31', freq='H')
n_samples = len(date_range)

# Create realistic consumption pattern
data = pd.DataFrame({
    'timestamp': date_range,
    'hour': date_range.hour,
    'day_of_week': date_range.dayofweek,
    'month': date_range.month,
    'is_weekend': (date_range.dayofweek >= 5).astype(int)
})

# Hourly pattern (typical German household)
hourly_pattern = np.array([
    300, 280, 250, 240, 250, 350, 550, 700,  # Night to morning
    600, 500, 450, 420, 430, 480, 550, 600,  # Midday
    700, 800, 750, 700, 650, 600, 550, 400   # Evening to night
])

data['consumption_base'] = hourly_pattern[data['hour']]

# Seasonal variation
seasonal_factor = 1.0 + 0.3 * np.sin(2 * np.pi * (data['month'] - 2) / 12)
data['consumption_seasonal'] = data['consumption_base'] * seasonal_factor

# Weekend effect
weekend_factor = data['is_weekend'].apply(lambda x: 0.85 if x else 1.0)
data['consumption_adjusted'] = data['consumption_seasonal'] * weekend_factor

# Add noise and temperature
data['consumption'] = data['consumption_adjusted'] + np.random.normal(0, 50, n_samples)
data['consumption'] = data['consumption'].clip(lower=50)
data['temperature'] = 15 + 8 * np.sin(2 * np.pi * (data['month'] - 2) / 12) + np.random.normal(0, 2, n_samples)

data.set_index('timestamp', inplace=True)

print(f"Dataset shape: {data.shape}")
print(f"Date range: {data.index.min()} to {data.index.max()}")
print(f"\nFirst 5 rows:")
print(data.head())

## 2. Prepare Data for LSTM (Sequence Format)

**Key Difference from Block 1:**
- Block 1: Each row = one prediction (features → target)
- Block 2: Sequences → predictions (lookback window → forecast horizon)

**LSTM Input Shape:** `(samples, time_steps, features)`
- `samples`: Number of training examples
- `time_steps`: Lookback window (e.g., 24 hours)
- `features`: Number of variables (consumption, temperature, hour, etc.)

In [ ]:
def create_sequences(data, lookback=24, forecast_horizon=1):
    """
    Convert time series to supervised learning format for LSTM.
    
    Parameters:
    -----------
    data : numpy array
        Time series data (shape: samples x features)
    lookback : int
        Number of past time steps to use as input (default: 24 hours)
    forecast_horizon : int
        Number of future time steps to predict (default: 1 hour)
    
    Returns:
    --------
    X : numpy array (samples, lookback, features)
        Input sequences
    y : numpy array (samples, forecast_horizon)
        Target sequences
    """
    X, y = [], []
    for i in range(len(data) - lookback - forecast_horizon + 1):
        X.append(data[i:i+lookback])  # Input: 24 hours
        y.append(data[i+lookback:i+lookback+forecast_horizon, 0])  # Target: consumption only
    return np.array(X), np.array(y)

# Select features for LSTM
feature_cols = ['consumption', 'temperature', 'hour', 'day_of_week', 'is_weekend']
data_features = data[feature_cols].values

# Normalize data (LSTM performs better with scaled data)
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(data_features)

print(f"Original data shape: {data_features.shape}")
print(f"Scaled data shape: {data_scaled.shape}")
print(f"Features: {feature_cols}")

### 2.1 Create Sequences for Single-Step Forecasting (1 hour ahead)

In [ ]:
# Create sequences: 24-hour lookback, 1-hour forecast
lookback = 24
forecast_horizon = 1

X_seq, y_seq = create_sequences(data_scaled, lookback=lookback, forecast_horizon=forecast_horizon)

print(f"Sequence dataset created:")
print(f"  X_seq shape: {X_seq.shape} (samples, lookback_hours, features)")
print(f"  y_seq shape: {y_seq.shape} (samples, forecast_hours)")
print(f"\nInterpretation:")
print(f"  - We have {X_seq.shape[0]:,} training examples")
print(f"  - Each example uses {X_seq.shape[1]} hours of history")
print(f"  - With {X_seq.shape[2]} features per hour")
print(f"  - To predict {y_seq.shape[1]} hour(s) ahead")

### 2.2 Train-Test Split (Chronological)

In [ ]:
# Split: 80% train, 20% test (chronological)
split_idx = int(len(X_seq) * 0.8)

X_train = X_seq[:split_idx]
y_train = y_seq[:split_idx]
X_test = X_seq[split_idx:]
y_test = y_seq[split_idx:]

print(f"Train set: {X_train.shape[0]:,} samples")
print(f"Test set:  {X_test.shape[0]:,} samples")
print(f"\nTrain/test split: {split_idx / len(X_seq) * 100:.0f}% / {(len(X_seq) - split_idx) / len(X_seq) * 100:.0f}%")

## 3. Build LSTM Model (Single-Step Forecasting)

**Architecture:**
- Input: (24, 5) - 24 hours of 5 features
- LSTM Layer 1: 64 units, return sequences (for stacking)
- Dropout: 20% (prevent overfitting)
- LSTM Layer 2: 32 units
- Dense: 16 units (ReLU activation)
- Output: 1 unit (next hour consumption)

In [ ]:
# Build LSTM model
lstm_model = Sequential([
    layers.LSTM(64, return_sequences=True, input_shape=(lookback, len(feature_cols)), name='lstm_1'),
    layers.Dropout(0.2, name='dropout_1'),
    layers.LSTM(32, name='lstm_2'),
    layers.Dropout(0.2, name='dropout_2'),
    layers.Dense(16, activation='relu', name='dense_1'),
    layers.Dense(1, name='output')  # Single output: next hour consumption
], name='LSTM_Single_Step')

# Compile model
lstm_model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

# Model summary
print(lstm_model.summary())

# Count parameters
total_params = lstm_model.count_params()
print(f"\nTotal trainable parameters: {total_params:,}")

## 4. Train LSTM Model

In [ ]:
# Define callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# Train model
print("Training LSTM model...\n")
history = lstm_model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

print("\n✅ Training complete!")

### 4.1 Visualize Training History

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss
axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('LSTM Training: Loss Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(history.history['mae'], label='Training MAE', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Validation MAE', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('LSTM Training: MAE Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final metrics
final_train_loss = history.history['loss'][-1]
final_val_loss = history.history['val_loss'][-1]
print(f"\nFinal Training Loss: {final_train_loss:.6f}")
print(f"Final Validation Loss: {final_val_loss:.6f}")
print(f"Overfitting check: {'⚠️ Possible overfitting' if final_train_loss < final_val_loss * 0.7 else '✅ Good generalization'}")

## 5. Evaluate LSTM on Test Set

In [ ]:
# Predict on test set
lstm_pred_test_scaled = lstm_model.predict(X_test, verbose=0)

# Inverse transform to original scale
# Need to reconstruct full feature matrix for inverse_transform
y_test_full = np.zeros((len(y_test), len(feature_cols)))
y_test_full[:, 0] = y_test.flatten()  # Consumption is first feature
y_test_original = scaler.inverse_transform(y_test_full)[:, 0]

lstm_pred_full = np.zeros((len(lstm_pred_test_scaled), len(feature_cols)))
lstm_pred_full[:, 0] = lstm_pred_test_scaled.flatten()
lstm_pred_original = scaler.inverse_transform(lstm_pred_full)[:, 0]

# Calculate metrics
lstm_mae = mean_absolute_error(y_test_original, lstm_pred_original)
lstm_rmse = np.sqrt(mean_squared_error(y_test_original, lstm_pred_original))
lstm_r2 = r2_score(y_test_original, lstm_pred_original)

print("="*60)
print("LSTM SINGLE-STEP FORECAST PERFORMANCE")
print("="*60)
print(f"MAE:  {lstm_mae:.2f} W")
print(f"RMSE: {lstm_rmse:.2f} W")
print(f"R²:   {lstm_r2:.3f}")
print("="*60)
print(f"\nInterpretation:")
print(f"  - Average prediction error: {lstm_mae:.0f} W")
print(f"  - Model explains {lstm_r2*100:.1f}% of variance")

## 6. Multi-Step Forecasting (24 Hours Ahead)

Now let's build a model that predicts the next 24 hours (not just 1 hour)

In [ ]:
# Create sequences for 24-hour forecast
forecast_horizon_multi = 24

X_seq_multi, y_seq_multi = create_sequences(
    data_scaled, 
    lookback=lookback, 
    forecast_horizon=forecast_horizon_multi
)

# Split
split_idx_multi = int(len(X_seq_multi) * 0.8)
X_train_multi = X_seq_multi[:split_idx_multi]
y_train_multi = y_seq_multi[:split_idx_multi]
X_test_multi = X_seq_multi[split_idx_multi:]
y_test_multi = y_seq_multi[split_idx_multi:]

print(f"Multi-step dataset:")
print(f"  X shape: {X_train_multi.shape} (samples, lookback, features)")
print(f"  y shape: {y_train_multi.shape} (samples, forecast_hours)")
print(f"\n  → Predicting {forecast_horizon_multi} hours ahead")

### 6.1 Build Multi-Step LSTM Model

In [ ]:
# Build multi-step LSTM
lstm_multi_model = Sequential([
    layers.LSTM(128, return_sequences=True, input_shape=(lookback, len(feature_cols)), name='lstm_1'),
    layers.Dropout(0.3, name='dropout_1'),
    layers.LSTM(64, return_sequences=True, name='lstm_2'),
    layers.Dropout(0.3, name='dropout_2'),
    layers.LSTM(32, name='lstm_3'),
    layers.Dropout(0.2, name='dropout_3'),
    layers.Dense(64, activation='relu', name='dense_1'),
    layers.Dense(forecast_horizon_multi, name='output')  # 24 outputs (next 24 hours)
], name='LSTM_Multi_Step')

lstm_multi_model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

print(lstm_multi_model.summary())
print(f"\nTotal parameters: {lstm_multi_model.count_params():,}")

### 6.2 Train Multi-Step LSTM

In [ ]:
# Train
early_stop_multi = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

print("Training multi-step LSTM model...\n")
history_multi = lstm_multi_model.fit(
    X_train_multi, y_train_multi,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop_multi],
    verbose=1
)

print("\n✅ Multi-step training complete!")

### 6.3 Evaluate Multi-Step LSTM

In [ ]:
# Predict
lstm_multi_pred_scaled = lstm_multi_model.predict(X_test_multi, verbose=0)

# Inverse transform
y_test_multi_full = np.zeros((len(y_test_multi), forecast_horizon_multi, len(feature_cols)))
y_test_multi_full[:, :, 0] = y_test_multi
y_test_multi_original = np.zeros_like(y_test_multi)
for i in range(len(y_test_multi)):
    y_test_multi_original[i] = scaler.inverse_transform(y_test_multi_full[i])[:, 0]

lstm_multi_pred_full = np.zeros((len(lstm_multi_pred_scaled), forecast_horizon_multi, len(feature_cols)))
lstm_multi_pred_full[:, :, 0] = lstm_multi_pred_scaled
lstm_multi_pred_original = np.zeros_like(lstm_multi_pred_scaled)
for i in range(len(lstm_multi_pred_scaled)):
    lstm_multi_pred_original[i] = scaler.inverse_transform(lstm_multi_pred_full[i])[:, 0]

# Metrics
lstm_multi_mae = mean_absolute_error(y_test_multi_original.flatten(), lstm_multi_pred_original.flatten())
lstm_multi_rmse = np.sqrt(mean_squared_error(y_test_multi_original.flatten(), lstm_multi_pred_original.flatten()))
lstm_multi_r2 = r2_score(y_test_multi_original.flatten(), lstm_multi_pred_original.flatten())

print("="*60)
print("LSTM MULTI-STEP FORECAST PERFORMANCE (24 HOURS)")
print("="*60)
print(f"MAE:  {lstm_multi_mae:.2f} W")
print(f"RMSE: {lstm_multi_rmse:.2f} W")
print(f"R²:   {lstm_multi_r2:.3f}")
print("="*60)

# Error by forecast horizon
mae_by_hour = []
for h in range(forecast_horizon_multi):
    mae_h = mean_absolute_error(y_test_multi_original[:, h], lstm_multi_pred_original[:, h])
    mae_by_hour.append(mae_h)

print(f"\nError analysis by forecast horizon:")
print(f"  1-hour ahead MAE:  {mae_by_hour[0]:.2f} W")
print(f"  6-hour ahead MAE:  {mae_by_hour[5]:.2f} W")
print(f"  12-hour ahead MAE: {mae_by_hour[11]:.2f} W")
print(f"  24-hour ahead MAE: {mae_by_hour[23]:.2f} W")

## 7. Visualization: Actual vs LSTM Predictions

In [ ]:
# Plot single-step predictions (first 7 days)
test_subset = slice(0, 7*24)
actual_single = y_test_original[test_subset]
pred_single = lstm_pred_original[test_subset]

fig, ax = plt.subplots(figsize=(14, 5))
hours = range(len(actual_single))
ax.plot(hours, actual_single, 'o-', label='Actual', linewidth=2, markersize=4, alpha=0.7)
ax.plot(hours, pred_single, 's--', label='LSTM Prediction', linewidth=2, markersize=4, alpha=0.7)
ax.set_xlabel('Hour')
ax.set_ylabel('Consumption (W)')
ax.set_title('LSTM Single-Step Forecast: First 7 Days of Test Set')
ax.legend()
ax.grid(True, alpha=0.3)

# Add MAE annotation
ax.text(0.98, 0.95, f'MAE: {lstm_mae:.1f} W', transform=ax.transAxes,
       ha='right', va='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

In [ ]:
# Plot multi-step forecast example (one 24-hour window)
example_idx = 10
actual_24h = y_test_multi_original[example_idx]
pred_24h = lstm_multi_pred_original[example_idx]

fig, ax = plt.subplots(figsize=(14, 5))
hours_24 = range(24)
ax.plot(hours_24, actual_24h, 'o-', label='Actual', linewidth=3, markersize=8, alpha=0.8)
ax.plot(hours_24, pred_24h, 's--', label='LSTM 24h Forecast', linewidth=3, markersize=8, alpha=0.8)
ax.fill_between(hours_24, pred_24h * 0.9, pred_24h * 1.1, alpha=0.2, label='±10% Confidence')
ax.set_xlabel('Hour Ahead')
ax.set_ylabel('Consumption (W)')
ax.set_title('LSTM Multi-Step Forecast: Next 24 Hours')
ax.set_xticks(range(0, 24, 3))
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Compare LSTM vs Block 1 Models

Let's compare LSTM performance with XGBoost from Block 1

In [ ]:
# Build simple XGBoost for comparison (same test set)
from xgboost import XGBRegressor

# Prepare flat features for XGBoost (flatten sequences)
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

# Train XGBoost
xgb_model = XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb_model.fit(X_train_flat, y_train.flatten(), verbose=False)

# Predict
xgb_pred_scaled = xgb_model.predict(X_test_flat)

# Inverse transform
xgb_pred_full = np.zeros((len(xgb_pred_scaled), len(feature_cols)))
xgb_pred_full[:, 0] = xgb_pred_scaled
xgb_pred_original = scaler.inverse_transform(xgb_pred_full)[:, 0]

# Metrics
xgb_mae = mean_absolute_error(y_test_original, xgb_pred_original)
xgb_rmse = np.sqrt(mean_squared_error(y_test_original, xgb_pred_original))
xgb_r2 = r2_score(y_test_original, xgb_pred_original)

print("XGBoost Performance (for comparison):")
print(f"  MAE:  {xgb_mae:.2f} W")
print(f"  RMSE: {xgb_rmse:.2f} W")
print(f"  R²:   {xgb_r2:.3f}")

In [ ]:
# Comparison table
comparison = pd.DataFrame({
    'Model': ['LSTM (1h ahead)', 'LSTM (24h ahead)', 'XGBoost (1h ahead)'],
    'MAE (W)': [lstm_mae, lstm_multi_mae, xgb_mae],
    'RMSE (W)': [lstm_rmse, lstm_multi_rmse, xgb_rmse],
    'R²': [lstm_r2, lstm_multi_r2, xgb_r2]
})

print("\n" + "="*70)
print("MODEL COMPARISON: LSTM vs XGBoost")
print("="*70)
print(comparison.to_string(index=False))
print("="*70)

# Winner for single-step
if lstm_mae < xgb_mae:
    improvement = (xgb_mae - lstm_mae) / xgb_mae * 100
    print(f"\n🏆 LSTM wins for single-step forecasting!")
    print(f"   {improvement:.1f}% better MAE than XGBoost")
else:
    improvement = (lstm_mae - xgb_mae) / lstm_mae * 100
    print(f"\n🏆 XGBoost wins for single-step forecasting!")
    print(f"   {improvement:.1f}% better MAE than LSTM")

print(f"\n✨ Key Insight:")
print(f"   LSTM can do multi-step forecasting (24h ahead) that XGBoost cannot easily do.")
print(f"   Multi-step MAE: {lstm_multi_mae:.2f} W - still very competitive!")

## 9. Error Analysis by Forecast Horizon

In [ ]:
# Plot MAE by forecast horizon
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(1, forecast_horizon_multi+1), mae_by_hour, 'o-', linewidth=2, markersize=6, color='steelblue')
ax.axhline(y=lstm_mae, color='red', linestyle='--', linewidth=2, label=f'Single-step MAE: {lstm_mae:.1f}W')
ax.set_xlabel('Forecast Horizon (hours ahead)')
ax.set_ylabel('MAE (W)')
ax.set_title('LSTM Forecast Error by Horizon')
ax.set_xticks(range(1, 25, 2))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nObservation:")
print(f"  - Error increases with forecast horizon (expected)")
print(f"  - Short-term (1-6h): MAE ≈ {np.mean(mae_by_hour[:6]):.1f} W")
print(f"  - Long-term (18-24h): MAE ≈ {np.mean(mae_by_hour[18:]):.1f} W")

## 10. Key Takeaways

### What We Learned:

1. **LSTMs Handle Sequences:** Unlike Block 1 models, LSTMs naturally process time series data
2. **Multi-Step Forecasting:** LSTM can predict multiple hours ahead (24h) in one shot
3. **Performance Trade-off:** LSTM may be similar to XGBoost for 1h ahead, but excels at longer horizons
4. **Memory Matters:** LSTM "remembers" patterns from 24+ hours ago (daily cycles)
5. **More Parameters = More Data Needed:** LSTM has ~100K parameters vs XGBoost's implicit structure

### When to Use LSTM vs Traditional ML:

**Use LSTM when:**
- Multi-step forecasting needed (next 24 hours)
- Long-term dependencies critical (weekly/seasonal patterns)
- Sequence structure matters (order of events)
- Large dataset available (LSTM needs more data)

**Use XGBoost/Random Forest when:**
- Single-step forecasting sufficient
- Limited data available
- Interpretability important (feature importance)
- Fast training/inference required

### Business Impact for Hochfrequenz Clients:

- **Grid Operations:** 24h-ahead load forecasting → better unit commitment, lower costs
- **Market Participation:** Multi-step forecasts → optimize day-ahead bidding
- **Customer Services:** Predict tomorrow's consumption → proactive notifications
- **Demand Response:** Forecast peak hours → schedule flexibility programs

---

### Next: Block 3 - Anomaly Detection with Autoencoders
We'll use similar neural network building blocks (encoder-decoder) to detect unusual consumption patterns (fraud, failures, anomalies)!